# 02 · Silver — limpo, tipado, deduplicado

Tudo o que a Silver faz tem uma **regra explícita**. O que viola a regra não some: vai para a **quarentena**.

| Tabela | Regras |
|---|---|
| clientes | `cliente_id` int; `estado` em maiúsculas; e-mail em minúsculas, inválido vira `NULL`; datas em 3 formatos; fica a versão mais recente por `cliente_id` |
| produtos | `preco` e `custo` > 0, senão quarentena |
| pedidos | tipos corretos; deduplica por `pedido_id`; **cliente inexistente → quarentena**; `itens` explodem em `itens_pedido` |
| avaliacoes | `nota` entre 1 e 5, senão quarentena; deduplica por `avaliacao_id` |

A Silver é reconstruída a partir de **toda** a Bronze (todos os lotes), então é segura para reexecutar.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))          # para importar a pasta common/
from common import lake_local as L, medalhao as M
from pyspark.sql import functions as F
spark = L.get_spark()                              # PySpark local: sem cluster
print("Spark", spark.version, "| master:", spark.sparkContext.master, "| storage:", L.STORAGE)

## Antes da função: faça na mão
Na aula de Spark você explodiu a coluna `itens`. Agora vamos fazer isso **a partir da Bronze**, que tem uma diferença: ela guarda os pedidos duplicados.
Se explodirmos antes de deduplicar, cada item duplicado conta duas vezes e o faturamento fica inflado.

In [ ]:
from pyspark.sql.window import Window

brz = spark.table("lake.bronze.pedidos")
print("Bronze:", brz.count(), "linhas |", brz.select("pedido_id").distinct().count(), "pedidos distintos")

# 1) deduplica por pedido_id, ficando com a versão mais recente
w = Window.partitionBy("pedido_id").orderBy(F.col("_ingest_ts").desc())
pedidos_unicos = brz.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
print("Duplicados descartados:", brz.count() - pedidos_unicos.count())

# 2) explode itens (posexplode guarda a posição do item no pedido)
itens = (pedidos_unicos
    .select("pedido_id", "cliente_id", "data_pedido", "status",
            F.posexplode("itens").alias("item_seq", "item"))
    .select("pedido_id", "cliente_id", "data_pedido", "status", "item_seq",
            F.col("item.produto_id").alias("produto_id"),
            F.col("item.qtd").alias("qtd"),
            F.col("item.preco_unit").alias("preco_unit"))
    .withColumn("valor_item", F.round(F.col("qtd") * F.col("preco_unit"), 2)))

print("pedidos:", pedidos_unicos.count(), "| itens:", itens.count())
itens.show(5, truncate=False)

### Checagens de qualidade nos itens
Mesmas três perguntas da aula (valores inválidos, chave duplicada, produto inexistente), mas agora quem falha **não some**: ganha um motivo e vai para a quarentena.

In [ ]:
produtos = (spark.table("lake.bronze.produtos")
            .select(F.col("produto_id").cast("long").alias("produto_id"))
            .distinct().withColumn("_existe", F.lit(True)))

valor_ok = (F.col("qtd") > 0) & (F.col("preco_unit") > 0) & F.col("produto_id").isNotNull()

itens_f = (itens.join(produtos, "produto_id", "left")
    .withColumn("motivo",
        F.when(F.coalesce(~valor_ok, F.lit(True)), "valor_invalido")
         .when(F.col("_existe").isNull(), "produto_inexistente")))

itens_ok  = itens_f.filter("motivo IS NULL").drop("motivo", "_existe")
itens_rej = itens_f.filter("motivo IS NOT NULL")

print("itens válidos:", itens_ok.count(), "| rejeitados:", itens_rej.count())
itens_rej.groupBy("motivo").count().show()

# chave composta única (pedido_id, item_seq) nos válidos
dup = itens_ok.groupBy("pedido_id", "item_seq").count().filter("count > 1").count()
print("chaves duplicadas:", dup, "->", "OK" if dup == 0 else "ERRO")

In [ ]:
# Salvar na Silver

itens_ok.writeTo("lake.silver.itens_pedido_aula").using("iceberg").createOrReplace()
spark.table("lake.silver.itens_pedido_aula").count()

In [ ]:
resumo = M.build_silver(spark)
resumo

## Reconciliação: tudo que entrou tem que estar em algum lugar
Regra de ouro: **entrada = saída + quarentena**. Se a conta não fecha, o pipeline está perdendo dado sem avisar.

In [ ]:
n_bronze = spark.table("lake.bronze.pedidos").select("pedido_id").distinct().count()
n_silver = spark.table("lake.silver.pedidos").count()
n_quar   = spark.table("lake.silver.quarentena").filter("origem = 'pedidos_cliente_orfao'").count()
print(f"pedidos : {n_bronze} = {n_silver} (silver) + {n_quar} (quarentena) -> {n_bronze == n_silver + n_quar}")

n_mao    = spark.table("lake.silver.itens_pedido_aula").count()
n_funcao = spark.table("lake.silver.itens_pedido").count()
print(f"itens   : feito na mão = {n_mao} | feito pela função = {n_funcao}")

## O que foi para a quarentena?

In [ ]:
spark.table("lake.silver.quarentena").groupBy("origem").count().show(truncate=False)
spark.table("lake.silver.quarentena").show(5, truncate=False)

## Data Quality: medir, não só limpar
`DQ` aplica regras e devolve um relatório. Guardar o relatório numa tabela é o começo da **observabilidade**.

In [ ]:
from common.dq import DQ
cli = spark.table("lake.silver.clientes")
ped = spark.table("lake.silver.pedidos")
av  = spark.table("lake.silver.avaliacoes")

r  = DQ(spark, cli, "silver.clientes").not_null("cliente_id").unique("cliente_id").not_null("email", "aviso")
r2 = DQ(spark, ped, "silver.pedidos").not_null("pedido_id").unique("pedido_id") \
       .in_set("status", ["pago","enviado","entregue","cancelado","devolvido"])
r3 = DQ(spark, av,  "silver.avaliacoes").unique("avaliacao_id").between("nota", 1, 5)

rel = r.report()
for x in (r2, r3):
    rel = rel.unionByName(x.report())
rel.show(truncate=False)
rel.writeTo("lake.silver.dq_resultados").using("iceberg").createOrReplace()

## Histórico de versões: a Silver tem memória
Cada escrita no Iceberg cria um **snapshot**: uma versão completa da tabela naquele instante. Nada é editado no lugar. Isso responde a perguntas de governança:
**o que mudou, quando, e dá para voltar?**

In [ ]:
spark.sql("""
    SELECT committed_at, snapshot_id, operation,
           summary['added-records']   AS linhas_adicionadas,
           summary['total-records']   AS total_de_linhas
    FROM lake.silver.itens_pedido_aula.snapshots
    ORDER BY committed_at
""").show(truncate=False)

Vamos simular um erro humano: alguém "limpa" os pedidos cancelados achando que eram lixo.
Guardamos o id da versão **antes** do estrago.

In [ ]:
TABELA = "lake.silver.itens_pedido_aula"

snap_bom = spark.sql(f"SELECT snapshot_id FROM {TABELA}.snapshots ORDER BY committed_at DESC LIMIT 1").first()[0]
antes = spark.table(TABELA).count()

spark.sql(f"DELETE FROM {TABELA} WHERE status = 'cancelado'")        # a regra errada

depois = spark.table(TABELA).count()
print(f"linhas: {antes} -> {depois}  (sumiram {antes - depois})")
print("snapshot bom:", snap_bom)

In [ ]:
# voltar no tempo (só ler)
spark.sql(f"SELECT count(*) AS linhas_na_versao_antiga FROM {TABELA} VERSION AS OF {snap_bom}").show()
spark.sql(f"SELECT count(*) AS linhas_na_versao_atual   FROM {TABELA}").show()

In [ ]:

spark.sql(f"CALL lake.system.rollback_to_snapshot('silver.itens_pedido_aula', {snap_bom})")
print("linhas depois do rollback:", spark.table(TABELA).count())

Se alguma regra aparece com status `AVISO` ou `ERRO`, ótimo: é o ponto de conversa. (Dica: e-mail inválido vira `NULL` na Silver, e a regra `not_null(email)` aparece como `AVISO`: o cliente continua existindo, só perdeu o contato.)

## Checkpoint
- [ ] Silver com 5 tabelas + `quarentena` + `dq_resultados`.
- [ ] Você consegue explicar por que um pedido foi para a quarentena.

➡️ Próximo: `03_gold.ipynb` (o foco da aula)